[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S07/2_Practica/en_clase/S07_04_roc_pr_desbalance_en_clase.ipynb)

# S07 · 04 — Curvas ROC y PR, y qué hacer con clases desbalanceadas · versión en clase

**Tiempo estimado:** 15 min en clase · **Sprint 7** · M3 (UD 3.4)

📖 *¿El código se hace largo? Léelo [explicado por escrito](https://github.com/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S07/2_Practica/codigo_explicado/S07_04_roc_pr_desbalance.md): recorrido, glosario y qué hace cada celda.* <!--codigo-explicado-->

**Contexto TurisData.** Imagina un alojamiento con clientes muy fieles donde **solo el 6 %**
de las reservas se cancela. Un modelo que diga siempre «no cancelará» acertaría el 94 %…
y sería completamente inútil. Cuando el evento que nos importa es **raro** (cancelaciones,
fraudes, averías), la exactitud engaña y hay que usar otras herramientas.

**Al terminar sabrás:**
1. Detectar la «paradoja de la exactitud».
2. Interpretar y calcular las curvas **ROC** (AUC) y **precisión–recall** (AP).
3. Aplicar tres remedios: pesos por clase, submuestreo y ajuste del umbral.

> **Nota honesta:** `reservas_turisdata.csv` tiene un 46 % de cancelaciones. Para este
> laboratorio **simulamos** un escenario raro conservando solo una parte de las canceladas.
> Es un recurso didáctico: en un proyecto real no se manipula la tasa de la población de test.

> 🏫 **Versión «en clase»** · ⏱ ≈ 15 min <!--reparto-->
>
> - Es **la parte de este cuaderno que se hace en el aula**, con el docente: ejercicios **2**.
> - Lo demás (ejercicios 🏠 y mini-reto) está en el [cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S07/2_Practica/para_casa/S07_04_roc_pr_desbalance.ipynb), para hacer en casa.


In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils import resample
from sklearn.base import clone
from sklearn.metrics import (roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
                             precision_score, recall_score, accuracy_score)

warnings.filterwarnings("ignore", category=UserWarning)
SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["temporada_alta"] = df["fecha_llegada"].dt.month.isin([1, 2, 3, 7, 8, 12]).astype(int)

# Escenario simulado: conservamos todas las no canceladas y una parte de las canceladas (≈ 6 %)
neg = df[df["cancelada"] == 0]
pos = df[df["cancelada"] == 1].sample(n=int(len(neg) * 0.064), random_state=SEMILLA)
raro = pd.concat([neg, pos]).sample(frac=1, random_state=SEMILLA).reset_index(drop=True)
print(f"Reservas: {len(raro)} · canceladas: {raro['cancelada'].sum()} ({raro['cancelada'].mean():.1%})")

columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = raro[columnas_num + columnas_cat], raro["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=SEMILLA, stratify=y)

prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), columnas_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), columnas_cat),
])

def nuevo_modelo(**kwargs):
    # clone(prep): cada modelo necesita su propia copia del preprocesado (si no, se pisarían entre sí)
    return Pipeline([("prep", clone(prep)), ("clf", LogisticRegression(max_iter=1000, **kwargs))])

## 1. La paradoja de la exactitud


## 2. Curva ROC y AUC

**Idea.** Para **cada umbral posible** medimos:
- **TPR** (tasa de verdaderos positivos = recall): cancelaciones detectadas,
- **FPR** (tasa de falsos positivos): reservas que se mantienen pero marcamos como riesgo.

La **curva ROC** dibuja TPR frente a FPR. Un modelo inútil va por la diagonal; uno bueno se
pega a la esquina superior izquierda. El **AUC** (área bajo la curva) resume: 0,5 = azar; 1 = perfecto.
Interpretación intuitiva: *la probabilidad de que una reserva cancelada reciba una puntuación
más alta que una que no se cancela*.

In [ ]:
base = nuevo_modelo().fit(X_train, y_train)
proba_base = base.predict_proba(X_test)[:, 1]

**Ejercicio 2 · 🏫 en clase.** Con `y_test` y `proba_base` calcula `fpr, tpr, _ = roc_curve(...)` y
`auc = roc_auc_score(...)`.

In [ ]:
# TODO: roc_curve(y_test, proba_base) devuelve tres arrays; roc_auc_score(y_test, proba_base)
...
assert len(fpr) == len(tpr) and fpr[0] == 0 and tpr[-1] == 1, "fpr y tpr deben ser la curva ROC completa"
assert 0.6 < auc < 0.95, "El AUC de este modelo debería estar entre 0,6 y 0,95"

plt.figure(figsize=(5.5, 5))
plt.plot(fpr, tpr, color="#2a6f97", lw=2, label=f"Regresión logística (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Azar (AUC = 0,50)")
plt.title("Curva ROC")
plt.xlabel("Tasa de falsos positivos (falsas alarmas)")
plt.ylabel("Tasa de verdaderos positivos (recall)")
plt.legend(loc="lower right")
plt.show()

---

## 🏁 Fin de la parte de clase

- Lo que queda (los ejercicios 🏠 y el mini-reto) está en el **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S07/2_Practica/para_casa/S07_04_roc_pr_desbalance.ipynb)**.
- Ahí también tienes, en cada ejercicio de casa, su celda plegada «🆘 Solución de referencia» por si te quedas atascado.
